In [2]:
#from estnltk import Text
#from estnltk.taggers import VabamorfTagger, VabamorfAnalyzer
#from estnltk_neural.taggers import StanzaSyntaxTagger
#from estnltk.converters import text_to_json
import sys, os
import re
import csv
import pandas as pd
import json
from tqdm import tqdm

In [3]:
pd.set_option('display.max_colwidth', None)
pd.set_option("display.show_dimensions", True)

In [12]:
RESULT_DIR = "../results/"

DATA_FILE = RESULT_DIR+ "n80_examples_large_v01/gpt_v03/" + "gpt_10K_b10_run01.csv"

BM_DIR = "../locative_adverbial/benchmarks/syntax_errors/v_kesksona/"

VKESK_FILE = BM_DIR + "gpt_10K_b10_run01_vkesksona.csv"


In [5]:
df1 = pd.read_csv(DATA_FILE, encoding="utf-8", sep=",")

## v-kesksõna

In [9]:
vk = df1[(df1["sentence"].str.contains("pankreases ja maksas")) | (df1["sentence"].str.contains("ühes tunnis pakutavad"))]

In [13]:
vk.to_csv(VKESK_FILE, encoding="utf-8", index=False, sep=",", quoting=csv.QUOTE_MINIMAL)

## Ühildumine

In [ ]:
import pandas as pd
from estnltk import Text
from estnltk.taggers import VabamorfTagger, VabamorfAnalyzer
from tqdm import tqdm
import os
import configparser
import json
import csv
import sys
import estnltk
import sqlite3

pd.set_option('display.max_colwidth', None)

In [ ]:
RESULTS_DIR = "../../results/"
EXAMPLE_FILE = "n80_examples_large_v01/gpt_v02/gpt_10K_b10_v01.csv"
BM_DIR = "../locative_adverbial/benchmarks/syntax_errors/v_kesksona/"
RESULT_FILE = BM_DIR + "gpt_10K_b10_v01_no_yhildumine.csv"

DB_FILE = ".........../drive_data/v33_koondkorpus_transaktsioonid_v04_2.db"


In [ ]:
morf_tagger = VabamorfAnalyzer(output_layer='morph_analysis')

In [ ]:
# database file path
filename = DB_FILE
# connecting with database
conn = sqlite3.connect(filename)
cur = conn.cursor()

In [ ]:
df = pd.read_csv(RESULTS_DIR+EXAMPLE_FILE, encoding="utf-8",  sep=",")
df = df[df["classification2"]=="no"]

In [ ]:
ending_mapping = {"in": ["s",1], "el": ["st",2], "adit":["i", 1], 'ill':["sse" ,3] , 
                  'all':["le",2] , 'abl': ["lt", 2], 'ad':["l", 1] }



end_df = pd.DataFrame([], columns = df.columns)

for i in tqdm(range(len(df))): #tqdm(range(1130, 1140)): #

    head_word = df.iloc[i]["form"]
    case = df.iloc[i]["morph_case"]
    head_id = df.iloc[i]["head_id"]
    head_ending = ending_mapping[case][0] # käände lõpp

    text = Text(df.iloc[i]["sentence"]).tag_layer(["words", "sentences"]) #.tag_layer('morph_analysis')
    morf_tagger.tag(text)
    words = text.words #.split(" ")

    head_loc = df.iloc[i]["head_loc"]
    head_idx = int(head_loc -1)
    prev_idx = int(head_idx -1) if head_idx != 1 else None
    next_idx = int(head_idx +1) if head_idx != len(words)-1 else None
    
    #prev_word = words[prev_idx].text if prev_idx is not None else None
    #next_word = words[next_idx].text if next_idx is not None else None
    
    ## kontroll: kas on kaks sõna järjest samas käändes 
    prev_case = False
    next_case = False
    prev_word = None
    next_word = None
    if prev_idx is not None:
        pre_forms = list(text.morph_analysis[prev_idx].form)
        #print(pre_forms)
        for f in pre_forms:
            if case in f or (case == 'ill' and "adt" in f)  or (case == 'adit' and "ill" in f):
                prev_case = True
                prev_word = text.morph_analysis[prev_idx].text
                break
    if next_idx is not None:
        next_forms = list(text.morph_analysis[next_idx].form)
        #print(next_forms)
        for f in next_forms:
            if case in f  or (case == 'ill' and "adt" in f)  or (case == 'adit' and "ill" in f):
                next_case = True
                next_word = text.morph_analysis[next_idx].text
                break


    # kontroll, kas eelmine/järgmine sõna on eraldi reana transaction tabelis olemas
    on_eraldi_rida = False
    try:
        if prev_case:
            query = f"SELECT * FROM transaction_row where head_id={head_id} and form='{prev_word}'"
            prev_res = pd.read_sql(query, conn)
            if len(prev_res) != 0:
                on_eraldi_rida = True
        if next_case:
            query = f"SELECT * FROM transaction_row where head_id={head_id} and form='{next_word}'"
            next_res = pd.read_sql(query, conn)
            if len(next_res) != 0:
                on_eraldi_rida = True
    except Exception as e:
        print(str(e))
    
    if on_eraldi_rida:
        end_df = pd.concat([end_df,df.iloc[[i]]], ignore_index=True)


In [ ]:
end_df.to_csv(RESULTS_DIR+RESULT_FILE, encoding="utf-8", index = False, sep=",", quoting=csv.QUOTE_MINIMAL)

In [ ]:
conn.close()